# Exploração quantitativa

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/01_exploracao_quantitativa.ipynb)
## Tipos de variáveis
Nominais distinguem; ordinais ordenam; quantitativas discretas contam; contínuas medem. Datas e identificadores têm papéis próprios. `dtype` não determina a escala conceitual.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
import numpy as np

dados = pd.read_csv("dados/documentos.csv")
pd.DataFrame(
    [["genero", "nominal"], ["ano", "temporal"],
     ["palavras", "quantitativa discreta"], ["id_documento", "identificador"]],
    columns=["variavel", "escala"],
)

## Frequências e proporções

Se $x_i$ é a categoria do documento $i$, a frequência absoluta da categoria
$k$ e sua proporção são:

$$
f_k = \sum_{i=1}^{n}\mathbf{1}(x_i=k),
\qquad
p_k = \frac{f_k}{n}.
$$

| Símbolo | Significado | Operação em Python |
|---|---|---|
| $n$ | total de documentos incluídos | `len(dados)` |
| $f_k$ | documentos cuja categoria é $k$ | `value_counts()` |
| $p_k$ | parcela do total na categoria $k$ | `value_counts(normalize=True)` |

A função indicadora $\mathbf{1}(x_i=k)$ vale 1 quando o documento pertence à
categoria e 0 caso contrário. Declare sempre $n$: documentos não medem
automaticamente intensidade, importância histórica ou quantidade de menções.

In [ ]:
frequencias_tema = dados["tema"].value_counts().rename("frequencia")
proporcoes_tema = dados["tema"].value_counts(normalize=True).rename("proporcao")
pd.concat([frequencias_tema, proporcoes_tema], axis=1)

## Centro, quartis e dispersão

Para uma variável quantitativa com valores $x_1,\ldots,x_n$, a média é:

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i.
$$

Depois de ordenar os valores, $x_{(1)}\leq\cdots\leq x_{(n)}$, a mediana é:

$$
\widetilde{x}=
\begin{cases}
x_{((n+1)/2)}, & n \text{ ímpar},\\[4pt]
\dfrac{x_{(n/2)}+x_{(n/2+1)}}{2}, & n \text{ par}.
\end{cases}
$$

O pandas calcula por padrão a variância amostral e o desvio-padrão amostral:

$$
s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2,
\qquad
s=\sqrt{s^2}.
$$

A média usa todos os valores e é sensível a extremos; a mediana depende da
posição ordenada; a moda é o valor de maior frequência. O denominador $n-1$
corresponde a `var(ddof=1)` e `std(ddof=1)`. Se o objetivo fosse descrever uma
população integral com denominador $n$, seria necessário declarar `ddof=0`.

In [ ]:
x = dados["palavras"]
resumo = pd.Series({
    "n": x.count(),
    "media": x.mean(),
    "mediana": x.median(),
    "moda": x.mode().iloc[0],
    "q1": x.quantile(0.25, interpolation="linear"),
    "q3": x.quantile(0.75, interpolation="linear"),
    "variancia_amostral": x.var(ddof=1),
    "desvio_padrao_amostral": x.std(ddof=1),
})
resumo

## Distribuição e valores extremos

O intervalo interquartil cobre a metade central dos valores ordenados:

$$
IQR=Q_3-Q_1.
$$

A regra usada pelo boxplot define dois limites:

$$
L_{\mathrm{inferior}}=Q_1-1{,}5\,IQR,
\qquad
L_{\mathrm{superior}}=Q_3+1{,}5\,IQR.
$$

Um caso fora desses limites é um candidato à inspeção, nunca uma exclusão
automática ou prova de erro. Quartis possuem convenções de cálculo diferentes;
neste notebook registramos explicitamente a interpolação linear usada pelo pandas.

In [ ]:
q1 = x.quantile(0.25, interpolation="linear")
q3 = x.quantile(0.75, interpolation="linear")
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

extremos = dados[
    (dados["palavras"] < limite_inferior)
    | (dados["palavras"] > limite_superior)
]
print("Limites:", limite_inferior, "a", limite_superior)
extremos[["id_documento", "palavras", "genero", "tema"]]

## Tabela de contingência

Se $A$ representa o gênero e $B$ o tema, a célula $n_{ij}$ conta os
documentos que pertencem simultaneamente à linha $i$ e à coluna $j$:

$$
n_{ij}=\sum_{r=1}^{n}\mathbf{1}(A_r=i \land B_r=j).
$$

A proporção por linha usa como denominador o total daquela linha:

$$
p_{j\mid i}=\frac{n_{ij}}{\sum_j n_{ij}}.
$$

Assim, contagens e proporções por linha respondem perguntas diferentes.
`normalize="index"` implementa $p_{j\mid i}$. Um padrão descritivo não é teste,
explicação causal ou evidência automática de associação histórica.

In [ ]:
contagens = pd.crosstab(dados["genero"], dados["tema"])
proporcoes_por_genero = pd.crosstab(
    dados["genero"], dados["tema"], normalize="index"
).round(3)
contagens, proporcoes_por_genero

## Atividade
Classifique variáveis, escolha medidas e denominadores, inspecione extremo e contingência. Separe descrição, interpretação e hipótese. Escreva aqui.